# RadReport-VL + Adaptive NeSy-Gen Colab

This notebook uses `VivaanGupta17/radreport-vl` as the report drafter while preserving its vision encoder, cross-attention bridge, decoder, and classifier code. The only added pieces are dataset/draft adapters plus the Adaptive NeSy-Gen PrimeKG/LTN/gate reasoning and evaluation stack.

Methodologically: RadReport-VL drafts the report; Adaptive NeSy-Gen audits claim-level clinical statements with retrieval grounding, PrimeKG paths, LTN constraint scores, consistency gating, revision traces, ablations, and publication metrics.

In [ ]:
import shutil, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

BRANCH = 'codex/lightweight-one-day'
shutil.rmtree('/content/NesyGENAAAI', ignore_errors=True)
shutil.rmtree('/content/radreport-vl', ignore_errors=True)
subprocess.run(['git', 'clone', '-b', BRANCH, 'https://github.com/FaezehMillerAI/NesyGENAAAI.git', '/content/NesyGENAAAI'], check=True)
subprocess.run(['git', 'clone', 'https://github.com/VivaanGupta17/radreport-vl.git', '/content/radreport-vl'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lightweight]'], cwd='/content/NesyGENAAAI', check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'accelerate', 'timm', 'datasets', 'tokenizers', 'einops',
    'omegaconf', 'hydra-core', 'pandas', 'Pillow', 'opencv-python', 'tqdm',
    'pyyaml', 'tensorboard', 'matplotlib', 'seaborn', 'nltk', 'rouge-score',
    'bert-score', 'pycocoevalcap',
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/radreport-vl'], check=True)
subprocess.run([sys.executable, 'scripts/prepare_radreport_vl_adapters.py', '--repo', '/content/radreport-vl'], cwd='/content/NesyGENAAAI', check=True)

## Configure paths

This uses the same manifest, MedSigLIP cache, and PrimeKG cache contract as the other AAAI notebooks. Keep `RADREPORT_EPOCHS` small for a smoke run; increase only after the full pipeline works.

In [ ]:
import json, os, subprocess, sys, torch, yaml
from pathlib import Path

RUN_DATASET = 'iuxray_official'  # or 'mimic_cxr_official' if configured
paths = json.loads(Path('/content/NesyGENAAAI/configs/colab_paths.json').read_text())[RUN_DATASET]
MANIFEST = Path(paths['manifest'])
MEDSIGLIP_CACHE = Path(paths['medsiglip_cache'])
PRIMEKG_CACHE = Path(paths['primekg_cache'])

RUN_ROOT = Path('/content/drive/MyDrive/aaai_2026_experiments/radreport_vl_adaptive_nesy_gen') / RUN_DATASET
RAD_CONFIG = RUN_ROOT / 'radreport_vl_manifest_config.yaml'
RAD_CKPT_DIR = RUN_ROOT / 'checkpoints'
DRAFTS_JSONL = RUN_ROOT / 'radreport_vl_test_drafts.jsonl'
NESY_SUITE_DIR = RUN_ROOT / 'nesy_ablation_suite'
PUBLICATION_DIR = RUN_ROOT / 'publication_metrics'
IMAGE_CACHE_DIR = Path('/content/radreport_vl_image_cache') / RUN_DATASET
RUN_ROOT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
for path in (MANIFEST, MEDSIGLIP_CACHE, PRIMEKG_CACHE):
    assert path.exists(), f'Missing required artifact: {path}'

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
RADREPORT_BATCH = 2 if gpu_gb >= 35 else 1
RADREPORT_EPOCHS = 2  # increase after smoke run; architecture is unchanged
print({'gpu': torch.cuda.get_device_name(0), 'gpu_gb': round(gpu_gb, 1), 'batch': RADREPORT_BATCH, 'epochs': RADREPORT_EPOCHS})

## Write a manifest-aware RadReport-VL config

The model architecture remains RadReport-VL's default ViT-B/16 + Q-Former + BioGPT. The config only points data/checkpoints to Drive, disables phase-2 RL for the one-day run, and sets `lambda_cls=0.0` unless you have real CheXpert labels in the manifest metadata.

In [ ]:
base_cfg = yaml.safe_load(Path('/content/radreport-vl/configs/mimic_cxr_config.yaml').read_text())
base_cfg['data']['root_dir'] = str(MANIFEST)
base_cfg['data']['dataset'] = 'mimic_cxr'
base_cfg['data']['num_workers'] = 4
base_cfg['training']['phase1']['batch_size'] = RADREPORT_BATCH
base_cfg['training']['phase1']['gradient_accumulation_steps'] = max(1, 8 // RADREPORT_BATCH)
base_cfg['training']['phase1']['num_epochs'] = RADREPORT_EPOCHS
base_cfg['training']['phase1']['lambda_cls'] = 0.0
base_cfg['training']['phase1']['use_scheduled_sampling'] = False
base_cfg['training']['phase2']['enabled'] = False
base_cfg['checkpointing']['output_dir'] = str(RAD_CKPT_DIR)
base_cfg['logging']['run_name'] = 'radreport_vl_adaptive_nesy_gen'
base_cfg['logging']['use_wandb'] = False
base_cfg['logging']['use_tensorboard'] = True
base_cfg['hardware']['device'] = 'cuda'
base_cfg['hardware']['num_workers'] = 4
RAD_CONFIG.write_text(yaml.safe_dump(base_cfg, sort_keys=False), encoding='utf-8')
print(RAD_CONFIG)
print('Architecture:', base_cfg['model']['encoder_type'], base_cfg['model']['bridge_type'], base_cfg['model']['decoder']['backbone'])

## Train RadReport-VL on the manifest

The adapter caches Drive-backed images to `/content` SSD using `RADREPORT_VL_IMAGE_CACHE_DIR`. This is data plumbing only; it does not alter the model.

In [ ]:
env = os.environ.copy()
env['RADREPORT_VL_IMAGE_CACHE_DIR'] = str(IMAGE_CACHE_DIR)
train_cmd = [
    sys.executable, 'scripts/train.py',
    '--config', str(RAD_CONFIG),
    '--data_root', str(MANIFEST),
]
subprocess.run(train_cmd, cwd='/content/radreport-vl', env=env, check=True)
CHECKPOINT = RAD_CKPT_DIR / 'best_model.pt'
if not CHECKPOINT.exists():
    checkpoints = sorted(RAD_CKPT_DIR.glob('epoch_*.pt'))
    assert checkpoints, f'No RadReport-VL checkpoints found in {RAD_CKPT_DIR}'
    CHECKPOINT = checkpoints[-1]
print('Using checkpoint:', CHECKPOINT)

## Export RadReport-VL test drafts

This step writes reference-free drafts. Test references remain outside inference and are loaded only later by the publication evaluator.

In [ ]:
draft_cmd = [
    sys.executable, 'scripts/generate_manifest_drafts.py',
    '--manifest', str(MANIFEST),
    '--config', str(RAD_CONFIG),
    '--checkpoint', str(CHECKPOINT),
    '--output', str(DRAFTS_JSONL),
    '--split', 'test',
    '--num-beams', '2',
    '--max-new-tokens', '180',
]
subprocess.run(draft_cmd, cwd='/content/radreport-vl', check=True)
print(DRAFTS_JSONL)

## Run Adaptive NeSy-Gen reasoning and ablation suite

This is where PrimeKG validation, LTN scores, consistency gates, revision traces, reasoning paths, graph controls, and verifier ablations are applied to RadReport-VL drafts.

In [ ]:
nesy_cmd = [
    sys.executable, 'scripts/run_experiments.py',
    '--manifest', str(MANIFEST),
    '--medsiglip-cache', str(MEDSIGLIP_CACHE),
    '--primekg-cache', str(PRIMEKG_CACHE),
    '--backend', 'replay',
    '--drafts-jsonl', str(DRAFTS_JSONL),
    '--output', str(NESY_SUITE_DIR),
    '--suite',
]
subprocess.run(nesy_cmd, cwd='/content/NesyGENAAAI', check=True)
print(NESY_SUITE_DIR)
print('\n'.join(str(path) for path in sorted(NESY_SUITE_DIR.glob('*.jsonl'))))

## Publication metrics

Set `RUN_PUBLICATION_EVAL=True` after the suite files are complete. This uses the official text/clinical metric packages and writes expert-review packets. It can be slower than the NeSy replay.

In [ ]:
RUN_PUBLICATION_EVAL = False
if RUN_PUBLICATION_EVAL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[eval]'], cwd='/content/NesyGENAAAI', check=True)
    eval_cmd = [
        sys.executable, 'scripts/evaluate_publication.py',
        '--manifest', str(MANIFEST),
        '--predictions-dir', str(NESY_SUITE_DIR),
        '--primekg-cache', str(PRIMEKG_CACHE),
        '--output-dir', str(PUBLICATION_DIR),
        '--baseline', 'drafting_only',
        '--bootstrap-samples', '1000',
    ]
    subprocess.run(eval_cmd, cwd='/content/NesyGENAAAI', check=True)
    print(PUBLICATION_DIR)